In [ ]:
from google.colab import drive

drive.mount('/content/drive', force_remount=True)
import joblib
import os

PROJECT_DIR = "/content/drive/MyDrive/EV_Battery_Digital_Twin"

extra_model = joblib.load(
    os.path.join(PROJECT_DIR, "extra_trees_model.pkl")
)

model_features = joblib.load(
    os.path.join(PROJECT_DIR, "feature_columns.pkl")
)

print("Model loaded:", type(extra_model).__name__)
print("Features:", model_features)

Mounted at /content/drive
Model loaded: ExtraTreesRegressor
Features: ['voltage_mean', 'voltage_std', 'voltage_min', 'voltage_max', 'voltage_range', 'current_mean', 'current_std', 'current_min', 'current_max', 'temperature_mean', 'temperature_std', 'temperature_max', 'temperature_rise']


In [ ]:
# ============================================================
# RELOAD EV BATTERY DIGITAL TWIN PROJECT
# ============================================================

import os
import pandas as pd
import numpy as np
import joblib

# ------------------------------------------------------------
# 1. Mount Google Drive
# ------------------------------------------------------------

from google.colab import drive
drive.mount('/content/drive')

# ------------------------------------------------------------
# 2. Project directory
# ------------------------------------------------------------

PROJECT_DIR = "/content/drive/MyDrive/EV_Battery_Digital_Twin"

print("Project directory:")
print(PROJECT_DIR)

print("\nFiles in project:")
print(os.listdir(PROJECT_DIR))

# ------------------------------------------------------------
# 3. Load ML dataset
# ------------------------------------------------------------

DATA_PATH = os.path.join(
    PROJECT_DIR,
    "ml_dataset.csv"
)

ml_dataset = pd.read_csv(DATA_PATH)

print("\nML dataset loaded")
print("Shape:", ml_dataset.shape)

print("\nColumns:")
print(ml_dataset.columns.tolist())

# ------------------------------------------------------------
# 4. Load Extra Trees model
# ------------------------------------------------------------

MODEL_PATH = os.path.join(
    PROJECT_DIR,
    "extra_trees_model.pkl"
)

extra_model = joblib.load(MODEL_PATH)

print("\nModel loaded:")
print(type(extra_model).__name__)

# ------------------------------------------------------------
# 5. Load exact model features
# ------------------------------------------------------------

FEATURE_PATH = os.path.join(
    PROJECT_DIR,
    "feature_columns.pkl"
)

model_features = joblib.load(
    FEATURE_PATH
)

print("\nModel features:")
for i, feature in enumerate(model_features, 1):
    print(f"{i}. {feature}")

print("\nFeature count:", len(model_features))

# ------------------------------------------------------------
# 6. Check sequences
# ------------------------------------------------------------

print("\nSequences:")
print(
    sorted(
        ml_dataset["sequence_id"].unique()
    )
)

print("\nSamples per sequence:")
print(
    ml_dataset.groupby(
        "sequence_id"
    ).size()
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project directory:
/content/drive/MyDrive/EV_Battery_Digital_Twin

Files in project:
['ml_dataset.csv', 'extra_trees_model.pkl', 'rul_summary.csv', 'streamlit.log', 'feature_columns.pkl', 'digital_twin_history.csv', 'dashboard.py']

ML dataset loaded
Shape: (408, 19)

Columns:
['sequence_id', 'cycle', 'voltage_mean', 'voltage_std', 'voltage_min', 'voltage_max', 'voltage_range', 'current_mean', 'current_std', 'current_min', 'current_max', 'temperature_mean', 'temperature_std', 'temperature_max', 'temperature_rise', 'duration_s', 'energy_Wh', 'SOH', 'local_cycle']

Model loaded:
ExtraTreesRegressor

Model features:
1. voltage_mean
2. voltage_std
3. voltage_min
4. voltage_max
5. voltage_range
6. current_mean
7. current_std
8. current_min
9. current_max
10. temperature_mean
11. temperature_std
12. temperature_max
13. temperature_rise

Feature count: 13

Sequences

In [ ]:
print("Checking Digital Twin variables...")

print("Dataset shape:", ml_dataset.shape)

print("\nSequences:")
print(sorted(ml_dataset["sequence_id"].unique()))

print("\nModel features:")
print(model_features)

print("\nNumber of features:", len(model_features))

print("\nModel type:")
print(type(extra_model).__name__)

Checking Digital Twin variables...
Dataset shape: (408, 19)

Sequences:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

Model features:
['voltage_mean', 'voltage_std', 'voltage_min', 'voltage_max', 'voltage_range', 'current_mean', 'current_std', 'current_min', 'current_max', 'temperature_mean', 'temperature_std', 'temperature_max', 'temperature_rise']

Number of features: 13

Model type:
ExtraTreesRegressor


In [ ]:
print("================================")
print("MODEL CHECK")
print("================================")

print("Model expects:",
      extra_model.n_features_in_,
      "features")

print("\nFeatures used by model:")

for i, feature in enumerate(
    extra_model.feature_names_in_,
    1
):
    print(f"{i}. {feature}")

print(
    "\nSaved feature list matches model:",
    list(model_features)
    == list(extra_model.feature_names_in_)
)

MODEL CHECK
Model expects: 13 features

Features used by model:
1. voltage_mean
2. voltage_std
3. voltage_min
4. voltage_max
5. voltage_range
6. current_mean
7. current_std
8. current_min
9. current_max
10. temperature_mean
11. temperature_std
12. temperature_max
13. temperature_rise

Saved feature list matches model: True


In [ ]:
print("================================")
print("DIGITAL TWIN VARIABLES")
print("================================")

print("Dataset:", ml_dataset.shape)

print("Model:",
      type(extra_model).__name__)

print("Features:",
      len(model_features))

print("Sequences:",
      sorted(ml_dataset["sequence_id"].unique()))

DIGITAL TWIN VARIABLES
Dataset: (408, 19)
Model: ExtraTreesRegressor
Features: 13
Sequences: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]


In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression


def update_digital_twin(
    current_row,
    history,
    model,
    feature_columns,
    eol_threshold=80.0,
    trend_window=20
):
    """
    Update the EV Battery Digital Twin for one new cycle.
    """

    # --------------------------------------------------------
    # 1. Extract exactly the 13 features used by the model
    # --------------------------------------------------------

    X_current = current_row[
        feature_columns
    ].to_frame().T

    # --------------------------------------------------------
    # 2. Estimate SOH
    # --------------------------------------------------------

    estimated_soh = model.predict(X_current)[0]

    estimated_soh = float(
        np.clip(estimated_soh, 0, 100)
    )

    # --------------------------------------------------------
    # 3. Add estimated SOH to current observation
    # --------------------------------------------------------

    new_row = current_row.copy()

    new_row["estimated_SOH"] = estimated_soh

    updated_history = pd.concat(
        [
            history,
            new_row.to_frame().T
        ],
        ignore_index=True
    )

    # --------------------------------------------------------
    # 4. Current cycle
    # --------------------------------------------------------

    current_cycle = float(
        current_row["cycle"]
    )

    # --------------------------------------------------------
    # 5. Estimate recent degradation rate
    # --------------------------------------------------------

    recent = updated_history.tail(
        min(
            trend_window,
            len(updated_history)
        )
    ).copy()

    recent = recent.dropna(
        subset=["estimated_SOH"]
    )

    if len(recent) >= 2:

        trend_model = LinearRegression()

        trend_model.fit(
            recent[["cycle"]],
            recent["estimated_SOH"]
        )

        degradation_rate = float(
            trend_model.coef_[0]
        )

    else:

        degradation_rate = np.nan

    # --------------------------------------------------------
    # 6. Estimate EOL cycle
    # --------------------------------------------------------

    if (
        np.isfinite(degradation_rate)
        and degradation_rate < 0
        and estimated_soh > eol_threshold
    ):

        predicted_eol_cycle = (
            current_cycle
            +
            (
                estimated_soh - eol_threshold
            )
            /
            abs(degradation_rate)
        )

    else:

        predicted_eol_cycle = np.nan

    # --------------------------------------------------------
    # 7. Calculate RUL
    # --------------------------------------------------------

    if np.isfinite(predicted_eol_cycle):

        rul = max(
            0,
            predicted_eol_cycle - current_cycle
        )

    else:

        rul = np.nan

    # --------------------------------------------------------
    # 8. Health status
    # --------------------------------------------------------

    if estimated_soh >= 90:

        health_status = "Healthy"

    elif estimated_soh >= 80:

        health_status = "Degrading"

    else:

        health_status = "Severely degraded"

    # --------------------------------------------------------
    # 9. Return complete Digital Twin state
    # --------------------------------------------------------

    return {
        "cycle": current_cycle,
        "estimated_SOH": estimated_soh,
        "degradation_rate": degradation_rate,
        "predicted_EOL_cycle": predicted_eol_cycle,
        "RUL": rul,
        "health_status": health_status,
        "history": updated_history
    }


print("Digital Twin update function created successfully.")

Digital Twin update function created successfully.


In [ ]:
seq5 = (
    ml_dataset[
        ml_dataset["sequence_id"] == 5
    ]
    .sort_values("cycle")
    .reset_index(drop=True)
)

print("Sequence 5 samples:", len(seq5))
print(
    "Cycle range:",
    seq5["cycle"].min(),
    "to",
    seq5["cycle"].max()
)

Sequence 5 samples: 107
Cycle range: 453 to 714


In [ ]:
history = seq5.iloc[:40].copy()

current_row = seq5.iloc[40]

result = update_digital_twin(
    current_row=current_row,
    history=history,
    model=extra_model,
    feature_columns=model_features,
    eol_threshold=80,
    trend_window=20
)

In [ ]:
print("================================")
print("DIGITAL TWIN SINGLE-CYCLE TEST")
print("================================")

print(
    "Current cycle:",
    result["cycle"]
)

print(
    "Estimated SOH:",
    round(result["estimated_SOH"], 2),
    "%"
)

print(
    "Degradation rate:",
    round(result["degradation_rate"], 4),
    "% SOH/cycle"
)

print(
    "Predicted EOL cycle:",
    round(result["predicted_EOL_cycle"], 2)
    if np.isfinite(result["predicted_EOL_cycle"])
    else "Not available"
)

print(
    "Estimated RUL:",
    round(result["RUL"], 2)
    if np.isfinite(result["RUL"])
    else "Not available",
    "cycles"
)

print(
    "Health status:",
    result["health_status"]
)

DIGITAL TWIN SINGLE-CYCLE TEST
Current cycle: 513.0
Estimated SOH: 97.3 %
Degradation rate: nan % SOH/cycle
Predicted EOL cycle: Not available
Estimated RUL: Not available cycles
Health status: Healthy


In [ ]:
# ============================================================
# INITIALIZE DIGITAL TWIN HISTORY
# ============================================================

seq5 = (
    ml_dataset[
        ml_dataset["sequence_id"] == 5
    ]
    .sort_values("cycle")
    .reset_index(drop=True)
)

initial_samples = 40

initial_history = seq5.iloc[
    :initial_samples
].copy()

# Estimate SOH for the initial history
initial_X = initial_history[
    model_features
]

initial_predictions = extra_model.predict(
    initial_X
)

initial_history[
    "estimated_SOH"
] = np.clip(
    initial_predictions,
    0,
    100
)

print("Initial Digital Twin history created.")

print(
    "Samples:",
    len(initial_history)
)

print(
    "Cycle range:",
    initial_history["cycle"].min(),
    "to",
    initial_history["cycle"].max()
)

print("\nFirst few estimates:")

display(
    initial_history[
        [
            "cycle",
            "SOH",
            "estimated_SOH"
        ]
    ].head()
)

Initial Digital Twin history created.
Samples: 40
Cycle range: 453 to 512

First few estimates:


,cycle,SOH,estimated_SOH
0,453,98.768121,98.843870
1,454,100.000000,99.647314
2,455,99.954288,99.682615
3,456,99.905359,99.690000
4,457,99.894141,99.667823


In [ ]:
history = initial_history.copy()

current_row = seq5.iloc[40]

result = update_digital_twin(
    current_row=current_row,
    history=history,
    model=extra_model,
    feature_columns=model_features,
    eol_threshold=80,
    trend_window=20
)

print("================================")
print("DIGITAL TWIN SINGLE-CYCLE TEST")
print("================================")

print(
    "Current cycle:",
    result["cycle"]
)

print(
    "Estimated SOH:",
    round(result["estimated_SOH"], 2),
    "%"
)

print(
    "Degradation rate:",
    round(result["degradation_rate"], 4),
    "% SOH/cycle"
)

print(
    "Predicted EOL cycle:",
    round(result["predicted_EOL_cycle"], 2)
    if np.isfinite(result["predicted_EOL_cycle"])
    else "Not available"
)

print(
    "Estimated RUL:",
    round(result["RUL"], 2)
    if np.isfinite(result["RUL"])
    else "Not available",
    "cycles"
)

print(
    "Health status:",
    result["health_status"]
)

DIGITAL TWIN SINGLE-CYCLE TEST
Current cycle: 513.0
Estimated SOH: 97.3 %
Degradation rate: -0.0453 % SOH/cycle
Predicted EOL cycle: 894.81
Estimated RUL: 381.81 cycles
Health status: Healthy


In [ ]:
history = initial_history.copy()

digital_twin_history = []

for i in range(initial_samples, len(seq5)):

    current_row = seq5.iloc[i]

    result = update_digital_twin(
        current_row=current_row,
        history=history,
        model=extra_model,
        feature_columns=model_features,
        eol_threshold=80,
        trend_window=20
    )

    digital_twin_history.append({
        "cycle": result["cycle"],
        "estimated_SOH": result["estimated_SOH"],
        "actual_SOH": current_row["SOH"],
        "degradation_rate": result["degradation_rate"],
        "predicted_EOL_cycle": result["predicted_EOL_cycle"],
        "RUL": result["RUL"],
        "health_status": result["health_status"]
    })

    # Update Digital Twin with newly observed cycle
    history = result["history"]

dt_history = pd.DataFrame(
    digital_twin_history
)

print("================================")
print("DIGITAL TWIN SIMULATION")
print("================================")

print("Initial history:", initial_samples)
print("Updated cycles:", len(dt_history))

display(
    dt_history.head(10).round(3)
)

DIGITAL TWIN SIMULATION
Initial history: 40
Updated cycles: 67


,cycle,estimated_SOH,actual_SOH,degradation_rate,predicted_EOL_cycle,RUL,health_status
0,513.0,97.305,97.322,-0.045,894.811,381.811,Healthy
1,514.0,97.255,97.261,-0.042,923.506,409.506,Healthy
2,515.0,97.191,97.195,-0.039,961.355,446.355,Healthy
3,516.0,97.169,97.135,-0.035,1008.836,492.836,Healthy
4,517.0,97.137,97.134,-0.030,1089.992,572.992,Healthy
5,518.0,97.185,97.196,-0.020,1372.090,854.090,Healthy
6,519.0,97.157,97.117,0.009,NaN,NaN,Healthy
7,520.0,97.172,97.148,-0.003,7378.560,6858.560,Healthy
8,521.0,97.048,96.986,-0.005,3917.952,3396.952,Healthy
9,522.0,97.013,97.003,-0.006,3257.245,2735.245,Healthy


In [ ]:
def calculate_stable_rul(
    history,
    current_cycle,
    current_soh,
    eol_threshold=80.0,
    trend_window=30,
    min_rate=0.05
):
    """
    Estimate RUL using a smoothed recent SOH degradation trend.

    min_rate prevents unrealistically large RUL values when
    the estimated degradation slope is almost zero.
    """

    recent = history.tail(trend_window).copy()

    recent = recent.dropna(
        subset=["estimated_SOH"]
    )

    if len(recent) < 5:
        return {
            "degradation_rate": np.nan,
            "predicted_EOL_cycle": np.nan,
            "RUL": np.nan
        }

    # --------------------------------------------------------
    # Smooth SOH before calculating trend
    # --------------------------------------------------------

    recent["SOH_smooth"] = (
        recent["estimated_SOH"]
        .rolling(
            window=5,
            min_periods=1
        )
        .mean()
    )

    # --------------------------------------------------------
    # Linear degradation trend
    # --------------------------------------------------------

    trend_model = LinearRegression()

    trend_model.fit(
        recent[["cycle"]],
        recent["SOH_smooth"]
    )

    degradation_rate = float(
        trend_model.coef_[0]
    )

    # --------------------------------------------------------
    # Ignore positive or extremely small degradation rates
    # --------------------------------------------------------

    if degradation_rate >= 0:

        return {
            "degradation_rate": degradation_rate,
            "predicted_EOL_cycle": np.nan,
            "RUL": np.nan
        }

    # Prevent extreme RUL due to nearly-zero slope
    effective_rate = min(
        degradation_rate,
        -min_rate
    )

    # --------------------------------------------------------
    # EOL prediction
    # --------------------------------------------------------

    predicted_eol_cycle = (
        current_cycle
        +
        (
            current_soh - eol_threshold
        )
        /
        abs(effective_rate)
    )

    rul = max(
        0,
        predicted_eol_cycle - current_cycle
    )

    return {
        "degradation_rate": degradation_rate,
        "predicted_EOL_cycle": predicted_eol_cycle,
        "RUL": rul
    }

In [ ]:
rul_result = calculate_stable_rul(
    history=result["history"],
    current_cycle=result["cycle"],
    current_soh=result["estimated_SOH"],
    eol_threshold=80,
    trend_window=30,
    min_rate=0.05
)

print("================================")
print("STABLE RUL TEST")
print("================================")

print(
    "Current cycle:",
    result["cycle"]
)

print(
    "Current SOH:",
    round(result["estimated_SOH"], 3),
    "%"
)

print(
    "Degradation rate:",
    round(
        rul_result["degradation_rate"],
        4
    ),
    "%/cycle"
)

print(
    "Predicted EOL:",
    round(
        rul_result["predicted_EOL_cycle"],
        2
    )
    if np.isfinite(
        rul_result["predicted_EOL_cycle"]
    )
    else "Not available"
)

print(
    "RUL:",
    round(
        rul_result["RUL"],
        2
    )
    if np.isfinite(
        rul_result["RUL"]
    )
    else "Not available",
    "cycles"
)

STABLE RUL TEST
Current cycle: 714.0
Current SOH: 79.639 %
Degradation rate: -0.1399 %/cycle
Predicted EOL: 711.42
RUL: 0 cycles


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

actual = dt_history["actual_SOH"]
estimated = dt_history["estimated_SOH"]

mae = mean_absolute_error(actual, estimated)

rmse = np.sqrt(
    mean_squared_error(actual, estimated)
)

r2 = r2_score(actual, estimated)

print("================================")
print("FINAL DIGITAL TWIN SOH RESULTS")
print("================================")

print(f"MAE  : {mae:.3f} %")
print(f"RMSE : {rmse:.3f} %")
print(f"R²   : {r2:.3f}")

FINAL DIGITAL TWIN SOH RESULTS
MAE  : 0.552 %
RMSE : 0.758 %
R²   : 0.990


In [ ]:
DT_HISTORY_PATH = os.path.join(
    PROJECT_DIR,
    "digital_twin_history.csv"
)

dt_history.to_csv(
    DT_HISTORY_PATH,
    index=False
)

print("Saved successfully:")
print(DT_HISTORY_PATH)

Saved successfully:
/content/drive/MyDrive/EV_Battery_Digital_Twin/digital_twin_history.csv
